### Scientific hypothesis

Phase 2 investigates whether metal-node chemistry, linker functionalization, linker size/geometry, and aromaticity provide additional information for predicting CO₂ uptake beyond the four pore-geometry descriptors used in Phase 1.

---

| Category | Candidate features | Why it may matter for CO₂ adsorption | Likely source |
| --- | --- | --- | --- |
| **1. Metal node / cluster** | Metal identity, atomic mass, oxidation state/valency, open metal sites | Controls electrostatic interaction and potential direct CO₂–metal binding | MOFID + external/derived descriptors; OMS may need MOF-specific structural analysis |
| **2. Linker functional groups** | `–NH₂`, `–OH`, `–F`, carboxylate, etc.; N/O/F/S counts | Changes polarity, electrostatics, and CO₂–framework interactions | RDKit |
| **3. Linker size & geometry** | Molecular weight, heavy-atom count, bond count, ring count, rotatable bonds, possibly linker length | Represents linker size, flexibility and connectivity | RDKit; true geometric length may require 3D information |
| **4. Aromaticity** | Aromatic atom count, aromatic ring count, aromatic fraction | Represents π-rich/aromatic character and linker rigidity | RDKit |

In [35]:
import pandas as  pd
df = pd.read_csv('/home/susan/mof-co2-adsorption/data/processed/data_clean_v2')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31234 entries, 0 to 31233
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   filename                  31234 non-null  object 
 1   lcd                       31234 non-null  float64
 2   pld                       31234 non-null  float64
 3   void_fraction             31234 non-null  float64
 4   surface_area_m2g          31234 non-null  float64
 5   mofid                     27706 non-null  object 
 6   CO2_uptake_0.01bar_molkg  31234 non-null  float64
 7   CO2_uptake_0.05bar_molkg  31234 non-null  float64
 8   CO2_uptake_0.1bar_molkg   31234 non-null  float64
 9   CO2_uptake_0.5bar_molkg   31234 non-null  float64
 10  CO2_uptake_2.5bar_molkg   31234 non-null  float64
dtypes: float64(9), object(2)
memory usage: 2.6+ MB


In [36]:
print(df['mofid'])

# count missing and avaliable MofIDs
print("total row:",len(df))
print('mofid available:', df['mofid'].notna().sum())
n_missing = df['mofid'].isna().sum()
pct_missing = df['mofid'].isna().mean()*100
print("Missing mofid:" ,n_missing)
print(f"Percentage missing:{pct_missing:.2f}%")





0        [O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...
1        [O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...
2        [O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Z...
3        COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1...
4        CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=...
                               ...                        
31229                                                  NaN
31230                                                  NaN
31231                                                  NaN
31232                                                  NaN
31233                                                  NaN
Name: mofid, Length: 31234, dtype: object
total row: 31234
mofid available: 27706
Missing mofid: 3528
Percentage missing:11.30%


In [48]:
# for chemistry experiment, we create a separate subset containing the 27,706 structures with MOFID:
df_chem = df[df['mofid'].notna()].copy()
df_chem.to_csv("df_chem.csv", index=False)
print('chemistry dataset:', df_chem.shape)

# investigate whether those 3,528 missing MOFIDs are randomly distributed or represent unusual structures
features = ["lcd", "pld", "void_fraction", "surface_area_m2g"]
df.groupby(df["mofid"].isna())[features].mean()


chemistry dataset: (27706, 11)


,lcd,pld,void_fraction,surface_area_m2g
mofid,,,,
False,8.810817,7.304591,0.648190,2563.596947
True,8.760771,7.254535,0.626014,2523.525312


---
Phase 2 – Chemistry dataset preparation: Of 31,234 cleaned MOF records, 27,706 (88.70%) contain MOFID chemical information and are available for chemistry-aware descriptor extraction. The remaining 3,528 records (11.30%) are retained in the master dataset but excluded from RDKit-based analysis.


---
**Check whether the missing MOFIDs introduce bias**: We checked whether removing MOFs without chemical information would obviously change the type of structures in our dataset. Based on the four geometric features, we did not see a major difference.

---

In [38]:
geometry_features = [
    "lcd",
    "pld",
    "void_fraction",
    "surface_area_m2g"
]

print('False means MOFID is present (27,706 MOFs), while True means MOFID is missing (3,528 MOFs).')

df.groupby(df["mofid"].isna())[geometry_features].agg(
    ["mean", "median", "std"]
).round(2)



False means MOFID is present (27,706 MOFs), while True means MOFID is missing (3,528 MOFs).


lcd                pld              void_fraction               \
       mean median   std  mean median   std          mean median   std   
mofid                                                                    
False  8.81   7.75  3.66  7.30   6.25  3.80          0.65   0.68  0.17   
True   8.76   8.25  3.53  7.25   6.25  3.75          0.63   0.68  0.18   

      surface_area_m2g                    
                  mean   median      std  
mofid                                     
False          2563.60  2515.55  1497.02  
True           2523.53  2642.65  1503.66

---
means, medians, and standard deviations are generally comparable between the two groups.
Therefore, based on these four structural descriptors, there is no obvious evidence that excluding the 11.3% without MOFID creates a strongly different geometric population.

---

SMILES = Simplified Molecular Input Line Entry System
Square brackets [ ] give special information about an atom.
[O-] means an oxygen atom carrying a −1 formal charge.
Lowercase c means aromatic carbon:
C → non-aromatic carbon
c → aromatic carbon
SMILES uses parentheses ( ) for branch from the preceding atom.
[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])...
`.`
This little dot is extremely important for separating the linker from the metal-containing part.
In SMILES, a dot `.` separates disconnected components.
For a very simple example:
`CCO.O`
RDKit reads this as two separate components:
`CCO`    .    `O`
 ↓           ↓
ethanol     water

In [39]:
# rdkit is the software package.
#Chem is the part of RDKit containing many of its chemistry tools.
from rdkit import Chem  # code
smiles = 'CCO'
mol = Chem.MolFromSmiles(smiles) 
#Take this SMILES text and
#convert it into a molecule that RDKit can understand and work with.”
print(type(smiles))
print(mol.GetNumAtoms())
for atom in mol.GetAtoms(): # gives you the atoms to iterate over
    print(atom.GetSymbol())
for bond in mol.GetBonds(): # To get bond type
    print(bond.GetBondType())

<class 'str'>
3
C
C
O
SINGLE
SINGLE


In [40]:
# But which atoms does each bond connect?
for bond in mol.GetBonds():
    print(bond.GetBeginAtom().GetSymbol(),
        bond.GetBondType(),
        bond.GetEndAtom().GetSymbol())

C SINGLE C
C SINGLE O


In [41]:
mol2= Chem.MolFromSmiles("CC(=O)O")
for bond in mol2.GetBonds():
    print(bond.GetBeginAtom().GetSymbol(),
        bond.GetBondType(),
        bond.GetEndAtom().GetSymbol())

C SINGLE C
C DOUBLE O
C SINGLE O


In [42]:
# count elements automatically
linker_smiles = "[O-]C(=O)c1ccc(cc1)C(=O)[O-]"
linker = Chem.MolFromSmiles(linker_smiles)
for atom in linker.GetAtoms():
    print(atom.GetSymbol())

elements = []

for atom in linker.GetAtoms():
    elements.append(atom.GetSymbol())

print(elements)
print("C:", elements.count("C"))
print("O:", elements.count("O"))
print("N:", elements.count("N"))
print("F:", elements.count("F"))

O
C
O
C
C
C
C
C
C
C
O
O
['O', 'C', 'O', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'O', 'O']
C: 8
O: 4
N: 0
F: 0


In [43]:
# Create RDKit molecule objects from SMILES barcodes
Tartrazine =Chem. MolFromSmiles('C1=CC(=CC=C1N=NC2C(=NN(C2=O)C3=CC=C(C=C3)S(=O)(=O)[O-])C(=O)[O-])S(=O)(=O)[O-].[Na+].[Na+].[Na+]')
elements =[]
for atom in Tartrazine.GetAtoms():
   elements.append(atom.GetSymbol())
print(elements)

for bond in Tartrazine.GetBonds():
    print(bond.GetBeginAtom().GetSymbol(),
        bond.GetBondType(),
        bond.GetEndAtom().GetSymbol())


['C', 'C', 'C', 'C', 'C', 'C', 'N', 'N', 'C', 'C', 'N', 'N', 'C', 'O', 'C', 'C', 'C', 'C', 'C', 'C', 'S', 'O', 'O', 'O', 'C', 'O', 'O', 'S', 'O', 'O', 'O', 'Na', 'Na', 'Na']
C AROMATIC C
C AROMATIC C
C AROMATIC C
C AROMATIC C
C AROMATIC C
C SINGLE N
N DOUBLE N
N SINGLE C
C SINGLE C
C DOUBLE N
N SINGLE N
N SINGLE C
C DOUBLE O
N SINGLE C
C AROMATIC C
C AROMATIC C
C AROMATIC C
C AROMATIC C
C AROMATIC C
C SINGLE S
S DOUBLE O
S DOUBLE O
S SINGLE O
C SINGLE C
C DOUBLE O
C SINGLE O
C SINGLE S
S DOUBLE O
S DOUBLE O
S SINGLE O
C AROMATIC C
C SINGLE C
C AROMATIC C


In [44]:
single_bonds = 0
double_bonds = 0
triple_bonds = 0
aromatic_bonds = 0

for bond in Tartrazine.GetBonds():
    bond_type = bond.GetBondType()

    if bond_type == Chem.BondType.SINGLE:
        single_bonds += 1

    elif bond_type == Chem.BondType.DOUBLE:
        double_bonds += 1

    elif bond_type == Chem.BondType.TRIPLE:
        triple_bonds += 1

    elif bond_type == Chem.BondType.AROMATIC:
        aromatic_bonds += 1

print("Single bonds:", single_bonds)
print("Double bonds:", double_bonds)
print("Triple bonds:", triple_bonds)
print("Aromatic bonds:", aromatic_bonds)

Single bonds: 13
Double bonds: 8
Triple bonds: 0
Aromatic bonds: 12


In [45]:
for atom in Tartrazine.GetAtoms():
    print(
        atom.GetSymbol(),
        "charge =", atom.GetFormalCharge())

C charge = 0
C charge = 0
C charge = 0
C charge = 0
C charge = 0
C charge = 0
N charge = 0
N charge = 0
C charge = 0
C charge = 0
N charge = 0
N charge = 0
C charge = 0
O charge = 0
C charge = 0
C charge = 0
C charge = 0
C charge = 0
C charge = 0
C charge = 0
S charge = 0
O charge = 0
O charge = 0
O charge = -1
C charge = 0
O charge = 0
O charge = -1
S charge = 0
O charge = 0
O charge = 0
O charge = -1
Na charge = 1
Na charge = 1
Na charge = 1
